In [3]:
from itertools import product
from tqdm import tqdm
import time
def generate_gene_sequences(file_name="gene_sequences20.txt", front="", end=""):
    bases = ['A', 'G', 'C', 'T']
    sequence_length = 16
    total_sequences = 4 ** sequence_length
    batch_size = 10000
    with open(file_name, "w") as file:
        buffer = []
        start_time = time.time()
        for i, sequence in enumerate(tqdm(product(bases, repeat=sequence_length), total=total_sequences, desc="Generating sequences")):
            formatted_sequence = f"{front}{''.join(sequence)}{end}\n"
            buffer.append(formatted_sequence)
            if (i + 1) % batch_size == 0:
                file.writelines(buffer)
                buffer = []
        if buffer:
            file.writelines(buffer)
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"All possible {sequence_length}-base gene sequences saved to {file_name}.")
    print(f"Elapsed time: {elapsed_time:.2f} seconds")
generate_gene_sequences(file_name="gene_sequences12-16-12.txt", front="TTTCGCACCAAC", end="AAGCGCCAAGTA")

Generating sequences:   7%|▋         | 291650730/4294967296 [01:41<23:09, 2881595.08it/s]  


KeyboardInterrupt: 

In [ ]:
import sqlite3
from itertools import product
from tqdm import tqdm
import time

def generate_gene_sequences_sql(db_name="gene_sequences.db", table_name="sequences", front="", end=""):
    bases = ['A', 'G', 'C', 'T']
    sequence_length = 16
    total_sequences = 4 ** sequence_length
    batch_size = 10000  # Batch insert for efficiency

    # Connect to SQLite database
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    # Create table if not exists
    cursor.execute(f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            sequence TEXT UNIQUE
        );
    """)
    conn.commit()

    start_time = time.time()
    buffer = []

    for i, sequence in enumerate(tqdm(product(bases, repeat=sequence_length), total=total_sequences, desc="Generating sequences")):
        formatted_sequence = f"{front}{''.join(sequence)}{end}"
        buffer.append((formatted_sequence,))

        # Insert in batches
        if (i + 1) % batch_size == 0:
            cursor.executemany(f"INSERT OR IGNORE INTO {table_name} (sequence) VALUES (?);", buffer)
            conn.commit()
            buffer = []

    # Insert remaining sequences
    if buffer:
        cursor.executemany(f"INSERT OR IGNORE INTO {table_name} (sequence) VALUES (?);", buffer)
        conn.commit()

    end_time = time.time()
    conn.close()

    print(f"All possible {sequence_length}-base gene sequences saved to {db_name} in table '{table_name}'.")
    print(f"Elapsed time: {end_time - start_time:.2f} seconds")

# Run the function
generate_gene_sequences_sql(db_name="gene_sequences12-16-12.db", table_name="sequences", front="TTTCGCACCAAC", end="AAGCGCCAAGTA")
